# MPR-Agent — graph-based multi-agent pipeline (ViNumQA)

Implementation of Nguyen et al., *"A Graph-Based Agent Approach to Numerical
Reasoning Question Answering"* ([VLSP 2025](https://aclanthology.org/2025.vlsp-1.29/)).
All logic lives in `agentic/` — see `README.md` beside this notebook for the
design and full ablation results. This notebook just runs it and prints PA/EA.

**Three ways to run this, only one applies at a time** — everything
environment-specific is marked as such below and stays commented out
otherwise, so switching just needs `MODEL` changed, no cells deleted:

| | `MODEL` | Needs | Environment-specific bits |
|---|---|---|---|
| **Local, Kaggle** | `Qwen3-4B` / `qwen3-4b-thinking` / `Gemma3-4B` | GPU + Internet | `!git clone` + `%cd` in the next cell — Settings -> Internet ON, Settings -> Accelerator = GPU. T4's 14.56 GB is tight: `LocalBackend` auto-retries at a smaller batch on OOM (may run slow on long prompts), see README.md. |
| **Local, Modal** | same three | A100-80GB Notebook | `!git clone` + `%cd` in the next cell, **Modal-labelled** variant — select GPU = A100-80GB in the Modal Notebook UI. No custom image needed (plain `transformers`, unlike the Unsloth/vLLM GRPO notebooks in `sft-grpo/`) — **untested on Modal specifically**, verify closely on first run. |
| **API** | any of the other eight (e.g. `DeepSeek-V4-Flash`) | `.env` with `API_KEY`/`BASE_URL` at the project root | None — runs on your own machine exactly like the DeepSeek-V4-Flash runs already done; leave both clone blocks commented |

Run the tests first either way: `pytest notebooks/vinumqa/graph-agent/tests -q`.

In [ ]:
# --- Kaggle only: clone the repo first, so .git/scorer.py/test.json/agentic/
# all come together with correct relative paths (uncomment both lines).
# Needs Internet ON in this notebook's Settings.
# -b chi: the OOM-retry fix in agentic/backends.py is only on this branch
# right now, not yet merged to main -- drop "-b chi" once it is.
!git clone -b chi https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText.git
%cd NumReasoning4VietnameseFinancialText

# --- Modal only (A100-80GB Notebook): same idea, plain repo clone -- no
# custom image needed here (unlike sft-grpo/'s Unsloth+vLLM notebooks, see
# modal/README.md), plain `transformers` runs fine on Modal's default image.
# Untested on Modal specifically -- watch the first real run closely.
# Select GPU = A100-80GB in the Notebook UI before running this cell.
# !git clone -b chi https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText.git
# %cd NumReasoning4VietnameseFinancialText

# Default output location -- repo-relative, fine for Kaggle and for local-
# machine/API runs (the working directory persists for the whole session).
GRAPH_AGENT_OUTPUT_DIR = "notebooks/vinumqa/graph-agent/outputs"

# --- Modal only: redirect checkpoints to a persistent Volume ---------------
# Modal Notebook containers are EPHEMERAL -- confirmed the hard way: a 497-
# sample run with no Volume attached died at 66/497 and every checkpointed
# sample was gone, not just the one in flight, because the whole container
# (including the git-cloned repo above) was destroyed with it. A Volume
# survives that. Attach one to this Notebook first -- Settings, same place
# as the custom image picker (see modal/README.md) -- note the mount path it
# reports, then uncomment below with that path.
#
# ALLOW_EPHEMERAL = True skips this check for a short `limit=` smoke test
# where losing everything on interruption is a minor loss; do not use it for
# a full 497-sample run.
# ALLOW_EPHEMERAL = False
# VOLUME_MOUNT = "/mnt/vol"   # <- set to wherever the Notebook reports it mounted
# if not ALLOW_EPHEMERAL:
#     import os
#     assert os.path.isdir(VOLUME_MOUNT), (
#         f"{VOLUME_MOUNT} not found -- attach a Volume to this Notebook first, "
#         f"or set ALLOW_EPHEMERAL = True to accept the risk (fine for a short "
#         f"limit= smoke test, not for a full run)."
#     )
#     GRAPH_AGENT_OUTPUT_DIR = f"{VOLUME_MOUNT}/graph-agent-outputs"

import sys
import time
from pathlib import Path

_here = Path.cwd()
ROOT = next((p for p in (_here, *_here.parents) if (p / ".git").exists()), None)
assert ROOT is not None, f"project root not found above {_here}"
HERE = ROOT / "notebooks" / "vinumqa" / "graph-agent"
sys.path.insert(0, str(HERE))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from agentic import AgentConfig, RunConfig, Runner, describe_backend

## Configuration

`MODEL` can be any of this repo's eleven baseline models — routed
automatically to local GPU or API (see `README.md`). `use_decomposition=False`
is the default: measured best on `DeepSeek-V4-Flash` (PA 0.7787, EA 0.8370,
full ablation table in `README.md`). Not yet re-verified on other models.

In [ ]:
MODEL = "qwen3-4b-thinking"
# MODEL = "Qwen3-4B" / "Gemma3-4B"   -- local, needs GPU
# MODEL = "DeepSeek-V4-Flash" / "gemma-3-27b-it" / "gemma-4-31B-it" / "gpt-oss-20b" /
#         "gpt-oss-120b" / "Llama-3.3-70B-Instruct" / "GLM-5.2" / "gpt-5-nano"  -- API, needs .env

# Kaggle "GPU T4 x2" only: True runs the parallel section further down and
# SKIPS the single-GPU cell right after this one; False is the reverse. Only
# one of the two ever actually executes, so "Run All" is safe either way --
# it will not silently burn the whole 497-sample run on 1 GPU while the
# second sits idle (that is what happens without this flag).
RUN_DUAL_GPU = True

agent_config = AgentConfig(
    model_subquery_gen=MODEL,
    model_subquery_ans=MODEL,
    model_planner=MODEL,
    model_fallback=MODEL,
    n_samples=15,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    prompt_lang="vi",
    vote_mode="canonical",
    use_prompt_ext=False,
    use_decomposition=False,   # best measured config on DeepSeek-V4-Flash -- NOT yet re-verified here
    max_workers_dataset=4,
    # rpm_limit=50, tpm_limit=100_000,   # GLM-5.2 on this endpoint -- see README.md
    # Kaggle T4 only: skip straight to a smaller n-sampling batch instead of
    # discovering it by OOMing first. Measured on a real run: batch=4 and
    # batch=2 BOTH OOM'd on the planner's ~3.9k token prompt, only batch=1
    # actually fit -- so 1, not a bigger "safer" guess, is what avoids every
    # wasted OOM retry on this GPU. Ignored on API models and irrelevant on a
    # roomier GPU (Modal A100-80GB), so leave commented there.
    local_max_batch_size=1 if describe_backend(MODEL) == "local" else None,
)

run_config = RunConfig(
    dataset_path="datasets/ViNumQA/origin/test.json",
    output_dir=GRAPH_AGENT_OUTPUT_DIR,   # repo-relative by default, Volume-backed on Modal (see setup cell)
    run_name=f"mpr-agent-{MODEL}",
    agent=agent_config,
    # limit=10,   # uncomment for a quick first check on a new MODEL/GPU combo before the full 497
)

runner = Runner(run_config)
print(f"MODEL={MODEL!r} -> {describe_backend(MODEL)} backend")
print(f"checkpoints -> {runner.checkpoint_path}")

In [ ]:
if not RUN_DUAL_GPU:
    started = time.time()
    df = runner.run(show_progress=True)
    scored, summary = runner.score(df)
    runner.save(scored, summary)

    print(f"\nPA: {summary['program_accuracy']}")
    print(f"EA: {summary['execution_accuracy']}")
    print(f"elapsed: {(time.time() - started) / 60:.1f} min")
else:
    print("RUN_DUAL_GPU=True -- skipping this cell, running the parallel section below instead.")

## Parallel run — Kaggle "GPU T4 x2" only

Controlled by `RUN_DUAL_GPU` in the config cell above — set it, don't skip
cells by hand, so this stays correct under "Run All" too. When this kernel
has two physical GPUs and `MODEL` is a local one: splits `test.json` in
half, runs one independent subprocess per GPU (same pattern this repo's
`sft-w-reasoning-trace-distill/qwen3-4b-eval-only-dual-gpu.ipynb` already
uses and has proven out), then merges both halves' scored results. Each
subprocess is self-contained (does not share this notebook's Python
namespace) and pins itself to one GPU via `CUDA_VISIBLE_DEVICES`, set
*before* `agentic`/`torch` is imported in that subprocess — setting it after
does not work, the CUDA context is already bound to whatever the process saw
at import time.

Roughly halves wall-clock vs the single-GPU cell above, since `LocalBackend`'s
OOM-retry (see README.md) applies independently within each subprocess/GPU —
this does not remove the need for it, it just runs two of it side by side.

In [ ]:
import json
import os
import queue
import subprocess
import threading

from agentic import load_dataset

if RUN_DUAL_GPU:
    N_GPUS = 2  # Kaggle "GPU T4 x2" -- change to however many physical GPUs this kernel has

    output_dir = Path(GRAPH_AGENT_OUTPUT_DIR)
    if not output_dir.is_absolute():
        output_dir = ROOT / output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    worker_script = r'''
import argparse, json, os, sys

p = argparse.ArgumentParser()
p.add_argument("--gpu", required=True)
p.add_argument("--root", required=True)
p.add_argument("--model", required=True)
p.add_argument("--input_json", required=True)
p.add_argument("--run_name", required=True)
p.add_argument("--output_dir", required=True)
args = p.parse_args()

# Must happen before agentic (-> torch/transformers) is imported: CUDA binds
# to whatever CUDA_VISIBLE_DEVICES says at import time, setting it later is a
# no-op and both processes would silently fight over GPU 0.
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
sys.path.insert(0, os.path.join(args.root, "notebooks", "vinumqa", "graph-agent"))

from agentic import AgentConfig, RunConfig, Runner

with open(args.input_json, encoding="utf-8") as f:
    samples = json.load(f)

# Same settings as the single-GPU config cell above -- keep the two in sync
# by hand if that cell's agent_config ever changes. local_max_batch_size=1:
# each GPU here is its own full T4 (Kaggle's "GPU T4 x2" does not split one
# T4's VRAM between the two), so the same OOM history applies per-process --
# measured on a real run, batch=4 and batch=2 both OOM'd on the planner's
# ~3.9k token prompt, only batch=1 fit -- go straight there.
agent_config = AgentConfig(
    model_subquery_gen=args.model, model_subquery_ans=args.model,
    model_planner=args.model, model_fallback=args.model,
    n_samples=15, temperature=0.6, top_p=0.95, top_k=20, prompt_lang="vi",
    vote_mode="canonical", use_prompt_ext=False, use_decomposition=False,
    local_max_batch_size=1,
)
run_config = RunConfig(
    dataset_path="datasets/ViNumQA/origin/test.json",
    output_dir=args.output_dir, run_name=args.run_name, agent=agent_config,
)
runner = Runner(run_config)
df = runner.run(samples=samples, show_progress=True)
scored, summary = runner.score(df)
runner.save(scored, summary)
print("[gpu %s] done: PA=%s EA=%s" % (
    args.gpu, summary["program_accuracy"], summary["execution_accuracy"]), flush=True)
'''

    worker_path = output_dir / "_dual_gpu_worker.py"
    worker_path.write_text(worker_script, encoding="utf-8")

    all_samples = load_dataset("datasets/ViNumQA/origin/test.json")
    chunks = [all_samples[i::N_GPUS] for i in range(N_GPUS)]  # interleaved, so both halves are the same size +/-1
    print(f"{len(all_samples)} sample(s) total -> " + ", ".join(f"gpu{i}: {len(c)}" for i, c in enumerate(chunks)))
else:
    print("RUN_DUAL_GPU=False -- skipping this cell, the single-GPU cell above already ran.")

In [ ]:
if RUN_DUAL_GPU:
    started = time.time()

    procs = []
    for gpu_id, chunk in enumerate(chunks):
        input_path = output_dir / f"_dual_gpu_chunk_{gpu_id}.json"
        input_path.write_text(json.dumps(chunk, ensure_ascii=False), encoding="utf-8")
        cmd = [
            sys.executable, str(worker_path),
            "--gpu", str(gpu_id),
            "--root", str(ROOT),
            "--model", MODEL,
            "--input_json", str(input_path),
            "--run_name", f"mpr-agent-{MODEL}-gpu{gpu_id}",
            "--output_dir", str(output_dir),
        ]
        env = dict(os.environ)  # each subprocess's own copy -- never touch this notebook's own os.environ
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)  # belt-and-suspenders; the worker also sets this itself before importing torch
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, env=env)
        procs.append((gpu_id, proc))

    def _stream(gpu_id, proc, q):
        for line in proc.stdout:
            q.put((gpu_id, line.rstrip()))
        q.put((gpu_id, None))

    out_q = queue.Queue()
    threads = [threading.Thread(target=_stream, args=(gid, proc, out_q), daemon=True) for gid, proc in procs]
    for t in threads:
        t.start()

    finished = 0
    while finished < len(threads):
        gid, line = out_q.get()
        if line is None:
            finished += 1
            continue
        print(f"[gpu {gid}] {line}")

    for gid, proc in procs:
        proc.wait()
        if proc.returncode != 0:
            print(f"[gpu {gid}] WARNING: exited with code {proc.returncode} -- its results below may be incomplete/missing.")

    print(f"\nboth workers finished, elapsed: {(time.time() - started) / 60:.1f} min")
else:
    print("RUN_DUAL_GPU=False -- skipping this cell, the single-GPU cell above already ran.")

In [ ]:
# --- merge both GPUs' scored results into the same `scored`/`summary` shape
# the single-GPU cell produces, so the error-analysis cell below works
# unchanged either way.
import pandas as pd

if RUN_DUAL_GPU:
    frames, summaries = [], []
    for gpu_id in range(N_GPUS):
        name = f"mpr-agent-{MODEL}-gpu{gpu_id}"
        results_csv = output_dir / f"{name}_results.csv"
        summary_json = output_dir / f"{name}_summary.json"
        assert results_csv.exists() and summary_json.exists(), (
            f"gpu{gpu_id} produced no output ({results_csv} / {summary_json} missing) "
            f"-- check that worker's log above for the actual error."
        )
        frames.append(pd.read_csv(results_csv))
        with open(summary_json, encoding="utf-8") as f:
            summaries.append(json.load(f))

    scored = pd.concat(frames, ignore_index=True)
    weights = [len(f) for f in frames]
    total = sum(weights)

    # Every summary stat here is itself a per-sample mean, so a count-weighted
    # average of the two halves' means is exactly the mean over the union --
    # not an approximation -- as long as the weights are the true sample counts.
    summary = {"program_accuracy": scored["pa_score"].mean(), "execution_accuracy": scored["ea_score"].mean(), "n": total}
    for key in ("oracle_pa", "oracle_ea", "oracle_coverage", "fallback_rate", "empty_rate", "mean_usable_candidates"):
        summary[key] = sum(s[key] * w for s, w in zip(summaries, weights)) / total

    print(f"PA: {summary['program_accuracy']}")
    print(f"EA: {summary['execution_accuracy']}")
else:
    print("RUN_DUAL_GPU=False -- skipping this cell, `scored`/`summary` already set by the single-GPU cell above.")

### Error analysis

`oracle@n` (best of the 15 sampled candidates per sample) splits where PA is
lost: `oracle_pa − PA` is a correct program that voting picked wrong
(*heuristic selection error*, fixable by a better vote); `1 − oracle_pa` is a
sample where none of the 15 candidates were ever correct (*systematic
reasoning error* — voting cannot fix this, only a better model/prompt can).

In [ ]:
pa = summary["program_accuracy"]
oracle = summary["oracle_pa"]
print(f"correct and selected    : {pa}")
print(f"generated but out-voted : {oracle - pa}   (heuristic selection error)")
print(f"never generated         : {1 - oracle}   (systematic reasoning error)")
print(f"\nfallback rate           : {summary['fallback_rate']}")
print(f"empty predictions       : {summary['empty_rate']}")

wrong = scored[(scored["pa_score"] == 0) & scored["consensus"].notna()]
if len(wrong):
    print(f"\nmean consensus on PA-wrong samples: {wrong['consensus'].mean()}")
    print(f"of which unanimous (consensus 1.0): {(wrong['consensus'] == 1.0).mean()}")